## ARTI 503 - Parallel Image Filtering - Stage 1

* Boundary handling : zero padding.
* Input : grayscale PNG, uint8 (values 0-255)
* Output : same size as input, uint8, saved as PNG

In [3]:
!pip install numpy pillow

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [4]:
import numpy as np
import time
from PIL import Image
import os

### Sequential Sharpening

In [5]:
KERNEL = [[0, -1, 0],
          [-1, 5, -1],
          [0, -1, 0]]

In [6]:
def sharpen_sequential(img):
    """Apply the 3x3 sharpening kernel to a 2D grayscale image (sequential)."""
    h, w = img.shape

    # Boundary handling: zero padding
    padded = np.zeros((h + 2, w + 2), dtype=np.float32)
    padded[1:h + 1, 1:w + 1] = img

    out = np.zeros((h, w), dtype=np.float32)

    # SEQUENTIAL nested loops over every output pixel (i, j)
    for i in range(h):
        for j in range(w):
            s = 0.0
            for ki in range(3):
                row = padded[i + ki]
                for kj in range(3):
                    s += KERNEL[ki][kj] * row[j + kj]
            out[i, j] = s

    return np.clip(out, 0, 255).astype(np.uint8)

In [7]:
def benchmark(image_path, repeats=5):
    img = np.array(Image.open(image_path), dtype=np.float32)
    times = []
    for r in range(repeats):
        start = time.perf_counter()
        result = sharpen_sequential(img)
        times.append(time.perf_counter() - start)
        print(f"  run {r+1}: {times[-1]:.4f} s")
    return result, times

In [9]:
if __name__ == "__main__":
    IMAGE_DIR = "test_images"
    REPEATS = 3

    images = [
        "C:/Users/zaina/Downloads/PCA_Stage1/test_images/test_images/small_256x256.png",
        "C:/Users/zaina/Downloads/PCA_Stage1/test_images/test_images/medium_512x512.png",
        "C:/Users/zaina/Downloads/PCA_Stage1/test_images/test_images/large_1024x1024.png",
        "C:/Users/zaina/Downloads/PCA_Stage1/test_images/test_images/xlarge_2048x2048.png",
        "C:/Users/zaina/Downloads/PCA_Stage1/test_images/test_images/xxlarge_4096x4096.png"
    ]

    for path in images:
        if not os.path.exists(path):
            continue 
        filename = os.path.basename(path)
        print(f"\n{filename}:")
        result, times = benchmark(path, repeats=REPEATS)
        print(f"  min = {min(times):.4f} s | avg = {sum(times)/len(times):.4f} s")
        
        Image.fromarray(result).save(f"sharpened_{filename}")



small_256x256.png:
  run 1: 0.1512 s
  run 2: 0.1745 s
  run 3: 0.1535 s
  min = 0.1512 s | avg = 0.1597 s

medium_512x512.png:
  run 1: 0.6467 s
  run 2: 0.6479 s
  run 3: 0.6461 s
  min = 0.6461 s | avg = 0.6469 s

large_1024x1024.png:
  run 1: 2.6841 s
  run 2: 2.6792 s
  run 3: 2.7534 s
  min = 2.6792 s | avg = 2.7056 s

xlarge_2048x2048.png:
  run 1: 10.9210 s
  run 2: 10.8200 s
  run 3: 10.6798 s
  min = 10.6798 s | avg = 10.8069 s

xxlarge_4096x4096.png:
  run 1: 43.1439 s
  run 2: 42.7678 s
  run 3: 42.7835 s
  min = 42.7678 s | avg = 42.8984 s
